# LogRAG — Log Anomaly Detection Benchmark

**Paper:** Jiang et al. (ISSRE 2024) — *Leveraging RAG-Enhanced Large Language Model for Semi-Supervised Log Anomaly Detection*  
**Architecture:** DeepSVDD (semi-supervised) + RAG post-processing with LLM  
**Dataset:** BGL  

---
### Kaggle Setup
1. Upload dataset vào Kaggle Dataset `logad-data`:
   - `BGL.log` → `/kaggle/input/logad-data/bgl/BGL.log`
2. **Enable GPU** (T4 hoặc P100)
3. **OpenAI API key** (optional):
   - Notebook → Add-ons → Secrets → thêm `OPENAI_API_KEY`
   - Hoặc set `USE_LOCAL_LLM = True` để dùng Mistral-7B
   - Nếu không có key: RAG chạy với majority vote (không gọi LLM)
4. Run All

**Pipeline:** BGL.log → Drain parse → SentenceTransformer embed → DeepSVDD train → anomaly candidates → RAG + LLM verify → metrics

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/rubyhcm/code-papers.git'
REPO_DIR = '/kaggle/working/code-papers'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print('Repo ready:', REPO_DIR)

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
# Install LogRAG requirements (pin versions from its requirements.txt)
!pip install -q \
    'langchain==0.2.14' \
    'langchain_community==0.2.12' \
    'langchain_openai==0.1.22' \
    'openai==1.42.0' \
    'scikit_learn==1.3.2' \
    'torch==1.13.0' \
    'tqdm==4.66.1' \
    'PyYAML==6.0.1' \
    sentence-transformers \
    drain3

print('Dependencies installed.')

In [ ]:
# ── 3. Config ─────────────────────────────────────────────────────────────────
import os, sys
import torch

LOGRAG_DIR = os.path.join(REPO_DIR, 'LogRAG')
sys.path.insert(0, LOGRAG_DIR)

BGL_LOG    = '/kaggle/input/logad-data/bgl/BGL.log'
OUTPUT_DIR = '/kaggle/working/lograg_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── LLM selection ─────────────────────────────────────────────────────────────
USE_LOCAL_LLM  = False  # True → Mistral-7B via HuggingFace
LOCAL_LLM_NAME = 'mistralai/Mistral-7B-Instruct-v0.1'

# Load OpenAI API key from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = api_key
    print('OpenAI API key loaded from Kaggle Secrets.')
except Exception:
    api_key = os.environ.get('OPENAI_API_KEY', '')
    if api_key:
        print('OpenAI API key loaded from environment.')
    else:
        print('No OpenAI API key — RAG will use retrieval-only fallback.')

print('Config OK')

## Step 1: Prepare Structured Log (Drain parse → CSV)

In [ ]:
# ── 4. Parse BGL with Drain → structured CSV ──────────────────────────────────
# LogRAG expects a structured CSV with columns: LineId, Label, Content, EventTemplate
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
import pandas as pd
from tqdm import tqdm

config = TemplateMinerConfig()
config.drain_sim_th = 0.5
config.drain_depth  = 4
miner = TemplateMiner(config=config)

records = []
print('Parsing BGL with Drain...')
with open(BGL_LOG, 'rb') as f:
    for i, raw in enumerate(tqdm(f, desc='Lines')):
        try:
            line = raw.decode('utf-8').strip()
        except:
            continue
        parts = line.split(' ')
        if len(parts) < 9:
            continue
        label   = parts[0]           # '-' = normal, else error type
        content = ' '.join(parts[8:])
        result  = miner.add_log_message(content)
        records.append({
            'LineId':        i + 1,
            'Label':         label,
            'Content':       content,
            'EventTemplate': result['template_mined'],
            'EventId':       f'E{result["cluster_id"]}',
        })

df = pd.DataFrame(records)
anom_rate = (df['Label'] != '-').mean()
print(f'BGL: {len(df):,} lines | anomaly rate: {anom_rate:.4f} | templates: {df["EventTemplate"].nunique()}')

# Save to LogRAG's dataset dir
bgl_dataset_dir = os.path.join(LOGRAG_DIR, 'dataset', 'BGL')
os.makedirs(bgl_dataset_dir, exist_ok=True)
structured_path = os.path.join(bgl_dataset_dir, 'bgl-example.log_structured.csv')
df.to_csv(structured_path, index=False)
print(f'Saved: {structured_path}')

## Step 2: Configure LogRAG

In [ ]:
# ── 5. Write config.yaml ──────────────────────────────────────────────────────
import yaml

llm_name = LOCAL_LLM_NAME if USE_LOCAL_LLM else 'gpt-3.5-turbo'

config_dict = {
    'is_train':    True,
    'is_rag':      True,
    'dataset_name': 'BGL',
    'train_ratio': 0.8,
    'window_size': 200,
    'window_time': 1800,

    # paths
    'log_structed_path': structured_path,
    'encoder_path':      os.path.join(LOGRAG_DIR, 'pretrained'),

    # LLM
    'api_key':  os.environ.get('OPENAI_API_KEY', 'NO_KEY'),
    'api_base': 'https://api.openai-proxy.org/v1',
    'llm_name': llm_name,

    # DeepSVDD
    'normal_class':      0,
    'is_pretrain':       False,
    'optimizer_name':    'adam',
    'lr':                0.0001,
    'n_epochs':          50,          # paper uses 150; reduce for speed
    'lr_milestones':     [25],
    'batch_size':        4096,
    'weight_decay':      0.0005,
    'device':            str(DEVICE),
    'n_jobs_dataloader': 0,

    # RAG
    'threshold':          0.8,
    'topk':               5,
    'prompt':             'prompt5',
    'persist_directory':  os.path.join(OUTPUT_DIR, 'ragdb-bgl'),
}

config_path = os.path.join(LOGRAG_DIR, 'config.yaml')
with open(config_path, 'w') as f:
    yaml.dump(config_dict, f, default_flow_style=False)

print('config.yaml written:')
!cat {config_path}

## Step 3: Prepare Pretrained Encoder

LogRAG uses a SentenceTransformer model as the encoder for DeepSVDD input.

In [ ]:
# ── 6. Download/verify encoder model ─────────────────────────────────────────
# LogRAG's encoder_path should point to a SentenceTransformer model directory.
# We use 'all-MiniLM-L6-v2' (small, fast) or the original 'distilbert-base-nli-mean-tokens'
from sentence_transformers import SentenceTransformer

pretrained_dir = os.path.join(LOGRAG_DIR, 'pretrained')
os.makedirs(pretrained_dir, exist_ok=True)

# Check if already downloaded
encoder_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
print(f'Downloading encoder: {encoder_model_name}')
encoder = SentenceTransformer(encoder_model_name, device=str(DEVICE))
encoder.save(pretrained_dir)
print(f'Encoder saved to: {pretrained_dir}')

## Step 4: Run LogRAG

In [ ]:
# ── 7. Run LogRAG main.py ─────────────────────────────────────────────────────
os.chdir(LOGRAG_DIR)
!python main.py 2>&1 | tee {OUTPUT_DIR}/lograg_run.log

print('\nLogRAG run complete.')

## Step 5: Collect Results

In [ ]:
# ── 8. Parse results from log and output/results.json ─────────────────────────
import re, json
import matplotlib.pyplot as plt
import numpy as np

lograg_output_dir = os.path.join(LOGRAG_DIR, 'output')
print('LogRAG output files:', os.listdir(lograg_output_dir) if os.path.exists(lograg_output_dir) else 'NOT FOUND')

# Try to load results.json from output/
results_json_path = os.path.join(lograg_output_dir, 'results.json')
lograg_metrics = {}
if os.path.exists(results_json_path):
    with open(results_json_path) as f:
        lograg_metrics = json.load(f)
    print('results.json:', json.dumps(lograg_metrics, indent=2))

# Parse metrics from run log
def parse_lograg_log(path):
    result = {}
    with open(path) as f:
        for line in f:
            p  = re.search(r'[Pp]recision[:\s]+([\d.]+)', line)
            r  = re.search(r'[Rr]ecall[:\s]+([\d.]+)',    line)
            f1 = re.search(r'[Ff]1[:\s]+([\d.]+)',        line)
            if p:  result['precision'] = float(p.group(1))
            if r:  result['recall']    = float(r.group(1))
            if f1: result['f1']        = float(f1.group(1))
    return result

run_metrics = parse_lograg_log(f'{OUTPUT_DIR}/lograg_run.log')
print('Parsed from run log:', run_metrics)

final_results = {
    'metadata': {
        'model':   'LogRAG',
        'paper':   'ISSRE 2024',
        'dataset': 'BGL',
        'llm':     llm_name,
        'svdd_epochs': config_dict['n_epochs'],
    },
    'bgl': run_metrics if run_metrics else lograg_metrics,
}

out_path = f'{OUTPUT_DIR}/lograg_results.json'
with open(out_path, 'w') as f:
    json.dump(final_results, f, indent=2)
print(json.dumps(final_results, indent=2))

In [ ]:
# ── 9. Visualize ──────────────────────────────────────────────────────────────
metrics = final_results.get('bgl', {})
if metrics:
    fig, ax = plt.subplots(figsize=(7, 4))
    names = ['Precision', 'Recall', 'F1']
    vals  = [metrics.get(k, 0) for k in ['precision', 'recall', 'f1']]
    bars  = ax.bar(names, vals, color=['#9C27B0', '#E91E63', '#FF5722'], alpha=0.85, width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=11)
    ax.set_title('LogRAG — BGL Results', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/lograg_results.png', bbox_inches='tight')
    plt.show()

# Show anomaly detection output CSV if exists
anomaly_csv = os.path.join(lograg_output_dir, 'anomaly_logs_detc_by_svdd.csv')
if os.path.exists(anomaly_csv):
    anom_df = pd.read_csv(anomaly_csv)
    print(f'\nSVDD anomaly candidates: {len(anom_df):,}')
    print(anom_df.head(5).to_string())

print(f'\nAll outputs saved to {OUTPUT_DIR}')

## Appendix: Run Without OpenAI Key

Nếu không có OpenAI API key, có thể:
1. Set `is_rag: False` trong config → chỉ chạy DeepSVDD
2. Set `USE_LOCAL_LLM = True` → dùng Mistral-7B (cần GPU ~14GB VRAM)
3. Chạy cell dưới để disable RAG và chỉ evaluate DeepSVDD

In [ ]:
# ── Optional: Run DeepSVDD only (no LLM) ─────────────────────────────────────
# Uncomment and run this cell if you want DeepSVDD-only baseline

# config_dict['is_rag'] = False
# with open(config_path, 'w') as f:
#     yaml.dump(config_dict, f)

# os.chdir(LOGRAG_DIR)
# !python main.py 2>&1 | tee {OUTPUT_DIR}/svdd_only_run.log
# print('DeepSVDD-only run complete.')
print('(Cell disabled — uncomment to run DeepSVDD without RAG)')